In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [2]:
data = pd.read_csv('LungCancerDataset.csv')

X = data.iloc[:, :17] #All columns except the 18th (Target)
y = data.iloc[:, 17]  # Target variable (18th column)

#Switching yes, no to 1,0
y = y.str.lower().map({'yes': 1, 'no': 0})

print(X)
print(y)

      AGE  GENDER  SMOKING  FINGER_DISCOLORATION  MENTAL_STRESS  \
0      68       1        1                     1              1   
1      81       1        1                     0              0   
2      58       1        1                     0              0   
3      44       0        1                     0              1   
4      72       0        1                     1              1   
...   ...     ...      ...                   ...            ...   
4995   32       0        1                     1              0   
4996   80       0        1                     1              1   
4997   51       1        0                     0              1   
4998   76       1        0                     1              0   
4999   33       0        1                     0              0   

      EXPOSURE_TO_POLLUTION  LONG_TERM_ILLNESS  ENERGY_LEVEL  IMMUNE_WEAKNESS  \
0                         1                  0     57.831178                0   
1                         1      

In [3]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
def get_data(model, X_train, X_test, y_train, y_test, model_name):

    #Cross Validation Score
    cross_val = cross_val_score(model, X_train, y_train).mean()
    
    # Training metrics
    y_train_pred = model.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    train_precision = precision_score(y_train, y_train_pred, zero_division=0)
    train_recall = recall_score(y_train, y_train_pred)
    train_f1 = f1_score(y_train, y_train_pred)
    
    # Testing metrics
    y_test_pred = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred, zero_division=0)
    test_recall = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    return {
        'model_name': model_name,
        'cross_validation': cross_val,
        'train_accuracy': train_accuracy,
        'train_precision': train_precision,
        'train_recall': train_recall,
        'train_f1': train_f1,
        'test_accuracy': test_accuracy,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'test_f1': test_f1
    }

results = []

In [5]:
#All diffrent options for model
C_vals = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]
kernels = ['linear', 'poly', 'rbf', 'sigmoid']

In [6]:
for kernel in kernels:
    for C_val in C_vals:
        model = SVC(kernel=kernel, C=C_val, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        model_name = f"{kernel}-{C_val}"
        
        result = get_data(model, X_train_scaled, X_test_scaled, y_train, y_test, model_name)
        
        print(result)
        results.append(result)

{'model_name': 'linear-0.0001', 'cross_validation': np.float64(0.5997333333333333), 'train_accuracy': 0.5997333333333333, 'train_precision': 0.0, 'train_recall': 0.0, 'train_f1': 0.0, 'test_accuracy': 0.5712, 'test_precision': 0.0, 'test_recall': 0.0, 'test_f1': 0.0}
{'model_name': 'linear-0.001', 'cross_validation': np.float64(0.8661333333333333), 'train_accuracy': 0.8890666666666667, 'train_precision': 0.875952875952876, 'train_recall': 0.8421052631578947, 'train_f1': 0.8586956521739131, 'test_accuracy': 0.8848, 'test_precision': 0.884313725490196, 'test_recall': 0.8414179104477612, 'test_f1': 0.8623326959847036}
{'model_name': 'linear-0.01', 'cross_validation': np.float64(0.8642666666666667), 'train_accuracy': 0.8712, 'train_precision': 0.8335517693315858, 'train_recall': 0.8474350433044637, 'train_f1': 0.8404360753221011, 'test_accuracy': 0.8784, 'test_precision': 0.8568773234200744, 'test_recall': 0.8600746268656716, 'test_f1': 0.8584729981378026}
{'model_name': 'linear-0.1', 'cro

In [7]:
results_df = pd.DataFrame(results)

In [8]:
print("Result Metrics:")
print(results_df)

Result Metrics:
        model_name  cross_validation  train_accuracy  train_precision  \
0    linear-0.0001          0.599733        0.599733         0.000000   
1     linear-0.001          0.866133        0.889067         0.875953   
2      linear-0.01          0.864267        0.871200         0.833552   
3       linear-0.1          0.882133        0.886667         0.844430   
4         linear-1          0.886667        0.888267         0.847240   
5        linear-10          0.886667        0.888533         0.847784   
6       linear-100          0.886667        0.888800         0.847882   
7      linear-1000          0.886933        0.888267         0.847240   
8     linear-10000          0.887200        0.890400         0.851160   
9      poly-0.0001          0.599733        0.599733         0.000000   
10      poly-0.001          0.599733        0.599733         0.000000   
11       poly-0.01          0.599733        0.618400         0.897727   
12        poly-0.1          0.80720

In [9]:
results_df.to_csv('SVM_data.csv', index = False) 